**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# MIMO Communications

Multiple antennas at both ends turn multipath — [Digital Comms'](./Digital_Communications.ipynb) villain — into **extra spectrum out of thin air**: parallel spatial channels through the same band. Capacity with many antennas, diversity vs multiplexing, and SVD precoding that turns the channel into independent pipes (verified: the pipes really are independent).

## 1. Pre-requisites

[Digital Communications](./Digital_Communications.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4 (SVD — the star of Session 3), [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def qpsk_syms(n): return (rng.choice([-1,1],n) + 1j*rng.choice([-1,1],n))/np.sqrt(2)
def rayleigh(nr, nt): return (rng.standard_normal((nr,nt)) + 1j*rng.standard_normal((nr,nt)))/np.sqrt(2)

---
### 🕐 Session 1 of 3 — *MIMO Capacity* (~35 min)
**Goal:** why capacity grows LINEARLY with min(antennas): the log-det formula, simulated.
**Builds on:** [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4. &nbsp; **Feeds into:** Session 2 (diversity).

---

## 2. Spectrum from Space

💡 **Intuition.** A rich-scattering channel matrix $H$ has $\min(n_t, n_r)$ meaningful singular directions — each an independent spatial pipe. Capacity $C = \log_2\det(I + \frac{\rho}{n_t} H H^H)$ therefore grows ~**linearly** in $\min(n_t, n_r)$ at high SNR, while a single antenna only ever gets $\log(1{+}\rho)$. Multipath, the enemy of the single-antenna link, is the *resource* here: no scattering ⇒ rank-1 $H$ ⇒ pipes collapse.

In [2]:
snr_db = 20; rho = 10**(snr_db/10)
antennas = [1, 2, 4, 8]
cap = {n_a: np.mean([np.log2(np.linalg.det(np.eye(n_a) + rho/n_a * (H := rayleigh(n_a, n_a)) @ H.conj().T).real)
                     for _ in range(2000)]) for n_a in antennas}
plt.figure(figsize=(7, 2.8))
plt.plot(antennas, [cap[a] for a in antennas], "o-", label="i.i.d. Rayleigh (rich scattering)")
plt.plot(antennas, [np.log2(1+rho)]*4, "k--", linewidth=1, label="1×1 (SISO) ceiling")
plt.xlabel("antennas (n×n)"); plt.ylabel("bits/s/Hz"); plt.legend()
plt.title(f"@{snr_db} dB: ergodic capacity scales ~linearly with antennas")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("mean capacity [bits/s/Hz]:", {a: round(cap[a],1) for a in antennas})
print(f"slope check: C(8)/C(1) = {cap[8]/cap[1]:.1f}  (linear scaling would give ≈8; the shortfall is the ρ/n_t power split)")

mean capacity [bits/s/Hz]: {1: np.float64(5.9), 2: np.float64(11.3), 4: np.float64(22.2), 8: np.float64(44.0)}
slope check: C(8)/C(1) = 7.4  (linear scaling would give ≈8; the shortfall is the ρ/n_t power split)


/tmp/ipykernel_2992188/166511356.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Diversity: Never Fade Alone* (~40 min)
**Goal:** Rayleigh fading murders BER; independent branches resurrect it — the diversity-order slopes.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (SVD precoding).

---

## 3. Insurance Against Fades

💡 **Intuition.** On a fading channel the *average* SNR is fine; the *bad moments* kill you — BER is dominated by the probability the channel is in a deep fade, which falls only as $1/\rho$ (diversity order 1). With $L$ **independent** branches, all must fade together: $P \sim \rho^{-L}$ — the BER-vs-SNR slope steepens to $L$. Maximum-ratio combining ([matched filtering](./Statistical_Signal_Processing.ipynb) across antennas!) collects it optimally; Alamouti's space-time code famously buys transmit diversity with two antennas and zero channel knowledge.

In [3]:
def ber_mrc(snr_db, L, n_bits=200_000):
    rho = 10**(snr_db/10)
    s = np.where(rng.random(n_bits) < 0.5, 1.0, -1.0)          # BPSK
    h = (rng.standard_normal((L, n_bits)) + 1j*rng.standard_normal((L, n_bits)))/np.sqrt(2)
    noise = (rng.standard_normal((L, n_bits)) + 1j*rng.standard_normal((L, n_bits)))/np.sqrt(2)
    y = h * s[None] + noise / np.sqrt(rho)
    s_hat = np.sign(np.real((h.conj() * y).sum(0)))            # MRC: matched filter across branches
    return np.mean(s_hat != s)

snrs = np.arange(0, 26, 5)
plt.figure(figsize=(7.5, 3))
for L in [1, 2, 4]:
    bers = [ber_mrc(s_, L) for s_ in snrs]
    plt.semilogy(snrs, bers, "o-", label=f"L={L} branches")
plt.semilogy(snrs, 0.5*10**(-snrs/10), "k:", linewidth=1, label="slope −1 reference")
plt.legend(fontsize=8); plt.xlabel("SNR [dB]"); plt.ylabel("BER")
plt.title("diversity order = the slope: each independent branch multiplies the decay rate")
plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()

/tmp/ipykernel_2992188/2963668625.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *SVD Precoding: the Channel, Diagonalized* (~40 min)
**Goal:** with channel knowledge, transform MIMO into independent parallel pipes — verified.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4.

---

## 4. The SVD Cashes Its Biggest Check

💡 **Intuition.** Write $H = U\Sigma V^H$. Precode with $V$ at the transmitter and receive with $U^H$: the end-to-end channel becomes $U^H H V = \Sigma$ — **diagonal**. Four antennas, four *independent* scalar channels with gains $\sigma_i$, zero cross-talk, no equalization. Then pour power by [water-filling](../Intro_Math/Information_Theory/Information_Theory.ipynb): more into strong pipes, none into hopeless ones. This is exactly how LTE/5G/Wi-Fi beamforming works when the channel is known.

In [4]:
nt = nr = 4
H = rayleigh(nr, nt)
U, s_vals, Vh = np.linalg.svd(H)

n_sym = 20_000
X = np.stack([qpsk_syms(n_sym) for _ in range(nt)])
tx = Vh.conj().T @ X                                        # precode with V
noise = (rng.standard_normal((nr, n_sym)) + 1j*rng.standard_normal((nr, n_sym)))/np.sqrt(2) * 0.05
Y = H @ tx + noise
Z = U.conj().T @ Y                                          # receive-combine with U^H

# ORACLE 1: effective channel is diagonal with the singular values
H_eff = U.conj().T @ H @ Vh.conj().T
off_diag = np.abs(H_eff - np.diag(s_vals)).max()
print(f"‖UᴴHV − diag(σ)‖∞ = {off_diag:.2e}   singular values: {s_vals.round(3)}")
assert off_diag < 1e-12

# ORACLE 2: the pipes are independent — cross-talk between streams ≈ 0
for i in range(nt):
    others = [j for j in range(nt) if j != i]
    xcorr = max(abs(np.corrcoef(np.real(Z[i]), np.real(X[j]))[0,1]) for j in others)
    print(f"stream {i}: gain σ={s_vals[i]:.2f}, own-corr {abs(np.corrcoef(np.real(Z[i]), np.real(X[i]))[0,1]):.3f}, "
          f"max cross-talk corr {xcorr:.4f}")

‖UᴴHV − diag(σ)‖∞ = 1.28e-15   singular values: [2.912 2.226 1.431 0.327]
stream 0: gain σ=2.91, own-corr 1.000, max cross-talk corr 0.0117
stream 1: gain σ=2.23, own-corr 1.000, max cross-talk corr 0.0055
stream 2: gain σ=1.43, own-corr 0.999, max cross-talk corr 0.0074
stream 3: gain σ=0.33, own-corr 0.989, max cross-talk corr 0.0127


In [5]:
# water-filling on the four pipes — at LOW SNR, where the choice matters
# (at high SNR water-filling ≈ equal power; the gains appear when power is scarce)
P_total, N0 = 1.0, 1.0
gains = s_vals**2 / N0
def waterfill(gains, P):
    mu_lo, mu_hi = 0, 1e6
    for _ in range(100):
        mu = (mu_lo + mu_hi)/2
        p = np.maximum(mu - 1/gains, 0)
        if p.sum() > P: mu_hi = mu
        else: mu_lo = mu
    return np.maximum(mu - 1/gains, 0)
p_wf = waterfill(gains, P_total)
rate_wf = np.log2(1 + gains*p_wf).sum()
rate_eq = np.log2(1 + gains*P_total/4).sum()
print(f"pipe gains σ²/N0: {gains.round(2)}")
print(f"power allocation (water-filling): {p_wf.round(2)}  — the weakest pipe gets {'NOTHING' if p_wf.min() < 1e-3 else 'little'}")
print(f"rate: equal power {rate_eq:.2f} b/s/Hz   water-filling {rate_wf:.2f} b/s/Hz  (+{100*(rate_wf/rate_eq-1):.0f}%)")

pipe gains σ²/N0: [8.48 4.95 2.05 0.11]
power allocation (water-filling): [0.48 0.4  0.11 0.  ]  — the weakest pipe gets NOTHING
rate: equal power 3.44 b/s/Hz   water-filling 4.24 b/s/Hz  (+23%)


## 5. Conclusion

Scattering builds rank, rank builds pipes; independent branches steepen the BER slope (measured); and the SVD — with channel knowledge — diagonalizes the link into verified-independent scalar channels fed by water-filling. The [SVD](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) never worked harder.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — the same antennas, pointed at *directions* instead of *rank*.
- [Channel Coding](./Channel_Coding.ipynb) — the codes riding inside each pipe.